In [1]:
import json
import re

import pandas as pd
import joblib
import requests

model = joblib.load("ipl_winner_model.pkl")
scaler = joblib.load("scaler.pkl")
team_strength = joblib.load("team_strength.pkl")
label_encoder = joblib.load("label_encoder.pkl")

In [2]:
def parse_jsonp(text, callback_name):
    pattern = rf"^{callback_name}\((.*)\);?$"
    match = re.search(pattern, text.strip(), flags=re.DOTALL)
    if not match:
        raise ValueError(f"Could not parse JSONP response for callback: {callback_name}")
    return json.loads(match.group(1))


In [4]:
def load_ipl_2026_fixtures():
    competition_url = "https://scores.iplt20.com/ipl/mc/competition.js"
    competition_text = requests.get(competition_url, timeout=30).text
    competition_data = parse_jsonp(competition_text, "oncomptetion")

    competition = next(
        c for c in competition_data["competition"]
        if c.get("CompetitionName") == "IPL 2026"
    )

    competition_id = competition["CompetitionID"]
    feed_source = competition.get(
        "feedsource",
        "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds"
    )

    schedule_url = f"{feed_source}/{competition_id}-matchschedule.js"
    schedule_text = requests.get(schedule_url, timeout=30).text
    schedule_data = parse_jsonp(schedule_text, "MatchSchedule")

    rows = []
    for match in schedule_data.get("Matchsummary", []):
        match_order = str(match.get("MatchOrder", ""))
        match_no_match = re.search(r"(\d+)", match_order)
        match_no = int(match_no_match.group(1)) if match_no_match else None

        rows.append({
            "match_no": match_no,
            "date": match.get("MatchDate"),
            "time": match.get("MatchTime"),
            "venue": match.get("GroundName"),
            "city": match.get("city"),
            "team1": match.get("FirstBattingTeamCode"),
            "team2": match.get("SecondBattingTeamCode"),
            "team1_full": match.get("FirstBattingTeamName"),
            "team2_full": match.get("SecondBattingTeamName")
        })

    fixtures_df = pd.DataFrame(rows)
    fixtures_df = fixtures_df.sort_values("match_no").reset_index(drop=True)
    return fixtures_df

In [5]:
fixtures = load_ipl_2026_fixtures()
print(f"Loaded {len(fixtures)} IPL 2026 fixtures from the official IPL feed.")
fixtures.head()

Loaded 70 IPL 2026 fixtures from the official IPL feed.


,match_no,date,time,venue,city,team1,team2,team1_full,team2_full
0,1,2026-03-28,19:30,M Chinnaswamy Stadium,Bengaluru,SRH,RCB,Sunrisers Hyderabad,Royal Challengers Bengaluru
1,2,2026-03-29,19:30,Wankhede Stadium,Mumbai,MI,KKR,Mumbai Indians,Kolkata Knight Riders
2,3,2026-03-30,19:30,ACA Stadium,Guwahati,RR,CSK,Rajasthan Royals,Chennai Super Kings
3,4,2026-03-31,19:30,New International Cricket Stadium,New Chandigarh,PBKS,GT,Punjab Kings,Gujarat Titans
4,5,2026-04-01,19:30,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,LSG,DC,Lucknow Super Giants,Delhi Capitals


In [6]:
def predict_match(team1, team2):

    input_df = pd.DataFrame([{
        "team1_strength": team_strength.get(team1, 0),
        "team2_strength": team_strength.get(team2, 0),
        "toss_win_match": 0,
        "h2h_team1_wins": 0,
        "team1_venue_strength": 0,
        "toss_bat": 0
    }])

    input_scaled = scaler.transform(input_df)

    proba = model.predict_proba(input_scaled)[0]
    class_labels = label_encoder.inverse_transform(range(len(proba)))
    proba_map = dict(zip(class_labels, proba))

    team1_raw = float(proba_map.get(team1, 0.0))
    team2_raw = float(proba_map.get(team2, 0.0))

    # Convert multiclass probabilities into a head-to-head chance between team1 and team2.
    pair_total = team1_raw + team2_raw
    if pair_total > 0:
        team1_chance = team1_raw / pair_total
        team2_chance = team2_raw / pair_total
    else:
        team1_chance = 0.5
        team2_chance = 0.5

    winner = team1 if team1_chance >= team2_chance else team2

    return {
        "winner_prediction": winner,
        "team1_win_chance": round(team1_chance * 100, 2),
        "team2_win_chance": round(team2_chance * 100, 2)
    }

In [7]:
predictions = fixtures.apply(
    lambda x: predict_match(x["team1_full"], x["team2_full"]),
    axis=1
)

predictions_df = pd.DataFrame(list(predictions))

for col in ["winner_prediction", "team1_win_chance", "team2_win_chance"]:
    fixtures[col] = predictions_df[col]

# Remove duplicate column names caused by repeated notebook runs.
fixtures = fixtures.loc[:, ~fixtures.columns.duplicated()]

output_df = fixtures[[
    "match_no",
    "date",
    "time",
    "team1",
    "team2",
    "team1_full",
    "team2_full",
    "winner_prediction",
    "team1_win_chance",
    "team2_win_chance",
    "venue",
    "city"
]].copy()

output_df = output_df.rename(columns={
    "match_no": "Match #",
    "date": "Date",
    "time": "Time (IST)",
    "team1": "Team 1",
    "team2": "Team 2",
    "team1_full": "Team 1 Name",
    "team2_full": "Team 2 Name",
    "winner_prediction": "Predicted Winner",
    "team1_win_chance": "Team 1 Win %",
    "team2_win_chance": "Team 2 Win %",
    "venue": "Venue",
    "city": "City"
})




In [8]:
def highlight_predicted_winner(row):
    styles = ["" for _ in row.index]
    winner = row["Predicted Winner"]
    if winner == row["Team 1 Name"]:
        styles[row.index.get_loc("Team 1 Win %")] = "background-color: #d8f3dc; font-weight: 700;"
    elif winner == row["Team 2 Name"]:
        styles[row.index.get_loc("Team 2 Win %")] = "background-color: #d8f3dc; font-weight: 700;"
    styles[row.index.get_loc("Predicted Winner")] = "background-color: #ffe8a1; font-weight: 700;"
    return styles


styled_table = (
    output_df.style
    .format({"Team 1 Win %": "{:.2f}%", "Team 2 Win %": "{:.2f}%"})
    .apply(highlight_predicted_winner, axis=1)
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("background-color", "#1f2937"), ("color", "white")]},
        {"selector": "td", "props": [("padding", "6px 10px")]}
    ])
    .hide(axis="index")
)

styled_table

Match #,Date,Time (IST),Team 1,Team 2,Team 1 Name,Team 2 Name,Predicted Winner,Team 1 Win %,Team 2 Win %,Venue,City
1,2026-03-28,19:30,SRH,RCB,Sunrisers Hyderabad,Royal Challengers Bengaluru,Sunrisers Hyderabad,75.24%,24.76%,M Chinnaswamy Stadium,Bengaluru
2,2026-03-29,19:30,MI,KKR,Mumbai Indians,Kolkata Knight Riders,Mumbai Indians,80.36%,19.64%,Wankhede Stadium,Mumbai
3,2026-03-30,19:30,RR,CSK,Rajasthan Royals,Chennai Super Kings,Chennai Super Kings,38.36%,61.64%,ACA Stadium,Guwahati
4,2026-03-31,19:30,PBKS,GT,Punjab Kings,Gujarat Titans,Gujarat Titans,0.58%,99.42%,New International Cricket Stadium,New Chandigarh
5,2026-04-01,19:30,LSG,DC,Lucknow Super Giants,Delhi Capitals,Lucknow Super Giants,99.90%,0.10%,Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium,Lucknow
6,2026-04-02,19:30,KKR,SRH,Kolkata Knight Riders,Sunrisers Hyderabad,Sunrisers Hyderabad,19.95%,80.05%,Eden Gardens,Kolkata
7,2026-04-03,19:30,CSK,PBKS,Chennai Super Kings,Punjab Kings,Chennai Super Kings,99.71%,0.29%,MA Chidambaram Stadium,Chennai
8,2026-04-04,15:30,DC,MI,Delhi Capitals,Mumbai Indians,Mumbai Indians,23.35%,76.65%,Arun Jaitley Stadium,Delhi
9,2026-04-04,19:30,GT,RR,Gujarat Titans,Rajasthan Royals,Gujarat Titans,98.60%,1.40%,Narendra Modi Stadium,Ahmedabad
10,2026-04-05,15:30,SRH,LSG,Sunrisers Hyderabad,Lucknow Super Giants,Lucknow Super Giants,22.24%,77.76%,Rajiv Gandhi International Stadium,Hyderabad
